Setup, Ingestion & Read Scoring
Description: Imports dependencies, loads preview_1000_pairs.csv, parses AffMat_design.xlsx, and scores the reads to create the required design assignment columns (True_Heavy_Design and True_Light_Design).

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pptx import Presentation
from pptx.util import Inches, Pt

# 1. Load sequence dataset
seq_df = pd.read_csv('../preview_1000_pairs.csv')

# 2. Define Smart Aligner function
def score_library_read(read_seq, rules_dict):
    if not isinstance(read_seq, str):
        return pd.Series(["No sequence", 0, None])
        
    best_design = "Unknown"
    min_errors = float('inf')
    associated_intended = 0
    
    for design_name, rules in rules_dict.items():
        wt_seq = rules['wt_sequence']
        allowed_muts = rules['allowed_mutations']
        design_len = len(wt_seq)
        
        if len(read_seq) < design_len:
            continue
            
        for i in range(len(read_seq) - design_len + 1):
            window = read_seq[i:i+design_len]
            intended = 0
            errors = 0
            
            for pos in range(design_len):
                if window[pos] != wt_seq[pos]:
                    if pos in allowed_muts and window[pos] in allowed_muts[pos]:
                        intended += 1
                    else:
                        errors += 1
            
            if errors < min_errors:
                min_errors = errors
                associated_intended = intended
                best_design = design_name
                
    if min_errors == float('inf'):
        return pd.Series(["Un-alignable", 0, None])
                
    return pd.Series([best_design, associated_intended, min_errors])

# 3. Load PWM design rules
def extract_full_pwm(filepath, sheet_name):
    df = pd.read_excel(filepath, sheet_name=sheet_name)
    all_indices = df[df.iloc[:, 1].notna()].index.tolist()
    design_start_indices = all_indices[0::2]
    
    gene_pwms = {}
    
    for i in range(len(design_start_indices)):
        start_idx = design_start_indices[i]
        end_idx = design_start_indices[i+1] if i + 1 < len(design_start_indices) else len(df)
        
        design_name = df.iloc[start_idx, 1]
        wt_aa_sequence = "".join(df.iloc[start_idx + 1, 4:].dropna().astype(str))
        
        wt_row_idx = start_idx + 3
        if wt_row_idx >= end_idx: continue
        
        wt_row = df.iloc[wt_row_idx]
        wt_marker_cols = wt_row[wt_row == 'WT'].index.tolist()
        
        pwm_dict = {}
        allowed_mutations = {}
        
        for col_name in wt_marker_cols:
            col_idx = df.columns.get_loc(col_name)
            offset = 1
            while (col_idx + offset) < len(df.columns):
                if pd.isna(df.iloc[wt_row_idx, col_idx + offset]):
                    break
                
                seq_position = (col_idx + offset) - 4
                
                aa_probs = {}
                allowed_aas = []
                for aa_row_offset in range(5, 25):
                    aa_label = str(df.iloc[start_idx + aa_row_offset, col_idx]).strip()
                    prob = df.iloc[start_idx + aa_row_offset, col_idx + offset]
                    
                    if pd.notna(prob) and isinstance(prob, (int, float)):
                        aa_probs[aa_label] = float(prob)
                        if prob > 0:
                            allowed_aas.append(aa_label)
                            
                if aa_probs:
                    pwm_dict[seq_position] = aa_probs
                if allowed_aas:
                    allowed_mutations[seq_position] = allowed_aas
                
                offset += 1
                
        gene_pwms[design_name] = {
            'wt_sequence': wt_aa_sequence,
            'pwm': pwm_dict,
            'allowed_mutations': allowed_mutations
        }
        
    return gene_pwms

heavy_pwm_rules = extract_full_pwm('../AffMat_design.xlsx', 'Heavy Chain Design')
light_pwm_rules = extract_full_pwm('../AffMat_design.xlsx', 'Light Chain Design')

# 4. Score reads to create alignment design columns
print("Aligning Heavy Chain reads...")
seq_df[['True_Heavy_Design', 'H_Intended_Mutations', 'H_True_Errors']] = seq_df['HAA'].apply(
    lambda x: score_library_read(x, heavy_pwm_rules)
)

print("Aligning Light Chain reads...")
seq_df[['True_Light_Design', 'L_Intended_Mutations', 'L_True_Errors']] = seq_df['LAA'].apply(
    lambda x: score_library_read(x, light_pwm_rules)
)

print("Dataset successfully loaded and scored with design assignments!")

Aligning Heavy Chain reads...
Aligning Light Chain reads...
Dataset successfully loaded and scored with design assignments!


KL-Divergence Design Bias Calculation
Description: Now that True_Heavy_Design and True_Light_Design exist in seq_df, this calculates the KL divergence across all CDR positions.

In [4]:
from scipy.stats import entropy

def calculate_pwm_kl_divergence(seq_df, rules_dict, chain_col, design_col, chain_name):
    kl_results = []
    aa_list = sorted(list("ACDEFGHIKLMNPQRSTVWY"))
    
    for gene_name, rules in rules_dict.items():
        pwm_target = rules['pwm']
        wt_seq = rules['wt_sequence']
        design_len = len(wt_seq)
        
        gene_reads = seq_df[seq_df[design_col] == gene_name]
        if len(gene_reads) == 0 or not pwm_target:
            continue
            
        pos_counts = {p: {aa: 0 for aa in aa_list} for p in pwm_target.keys()}
        total_depth = 0
        
        for _, row in gene_reads.iterrows():
            read_seq = row[chain_col]
            read_count = row['read_count']
            if not isinstance(read_seq, str) or len(read_seq) < design_len:
                continue
                
            best_window = None
            min_err = float('inf')
            for i in range(len(read_seq) - design_len + 1):
                window = read_seq[i:i+design_len]
                errs = sum(1 for pos in range(design_len) if window[pos] != wt_seq[pos])
                if errs < min_err:
                    min_err = errs
                    best_window = window
                    
            if best_window:
                total_depth += read_count
                for pos in pwm_target.keys():
                    if pos < len(best_window):
                        res = best_window[pos]
                        if res in pos_counts[pos]:
                            pos_counts[pos][res] += read_count
                            
        if total_depth == 0:
            continue
            
        epsilon = 1e-6
        for pos, intended_dict in pwm_target.items():
            p_intended = np.array([intended_dict.get(aa, 0.0) for aa in aa_list])
            p_observed = np.array([pos_counts[pos][aa] for aa in aa_list]) / total_depth
            
            p_intended = np.clip(p_intended, epsilon, 1.0)
            p_intended /= p_intended.sum()
            
            p_observed = np.clip(p_observed, epsilon, 1.0)
            p_observed /= p_observed.sum()
            
            kl_div = entropy(p_observed, p_intended)
            
            kl_results.append({
                'Chain': chain_name,
                'Gene': gene_name,
                'Position': pos + 1,
                'WT_AA': wt_seq[pos] if pos < len(wt_seq) else '-',
                'KL_Divergence': round(kl_div, 4)
            })
            
    return pd.DataFrame(kl_results)

print("Calculating KL Divergence across Heavy and Light CDR designs...")
heavy_kl_df = calculate_pwm_kl_divergence(seq_df, heavy_pwm_rules, 'HAA', 'True_Heavy_Design', 'Heavy')
light_kl_df = calculate_pwm_kl_divergence(seq_df, light_pwm_rules, 'LAA', 'True_Light_Design', 'Light')

kl_summary_df = pd.concat([heavy_kl_df, light_kl_df], ignore_index=True)

print("\nPositions with Highest Synthesis Bias (Top KL Divergence):")
print(kl_summary_df.sort_values(by='KL_Divergence', ascending=False).head(10))

Calculating KL Divergence across Heavy and Light CDR designs...

Positions with Highest Synthesis Bias (Top KL Divergence):
     Chain      Gene  Position WT_AA  KL_Divergence
309  Light  IGLV2-14        24     G        13.8183
275  Light  IGLV1-44        36     W        13.8174
113  Heavy  IGHV3-30        59     Y        13.8152
115  Heavy  IGHV4-39        28     S        13.8152
212  Light  IGKV3-20        34     L        13.8152
71   Heavy  IGHV3-15        34     M        13.8152
250  Light  IGLV1-40        35     V        13.8152
81   Heavy  IGHV3-15        61     D        13.8152
321  Light  IGLV2-14        36     S        13.8152
320  Light  IGLV2-14        35     V        13.8152


ESM-2 Language Model Pseudo-Perplexity Scoring
Description: Loads Meta's esm2_t6_8M_UR50D transformer model to evaluate sequence log-likelihood. Sequences with lower perplexity scores are predicted to possess superior structural stability and wild-type scaffold compatibility.

In [5]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

print("Loading Meta ESM-2 transformer model...")
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)
model.eval()

def compute_esm2_perplexity(sequence):
    """
    Computes average sequence pseudo-perplexity using ESM-2.
    Lower score = higher predicted sequence stability and structural fitness.
    """
    if not isinstance(sequence, str) or len(sequence) == 0:
        return np.nan
        
    inputs = tokenizer(sequence, return_tensors="pt")
    input_ids = inputs["input_ids"]
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits
        
    log_probs = torch.log_softmax(logits, dim=-1)
    target_log_probs = log_probs[0, range(1, input_ids.shape[1] - 1), input_ids[0, 1:-1]]
    
    nll = -torch.mean(target_log_probs).item()
    return round(np.exp(nll), 3)

# Batch score unique sequences for speed
print("Evaluating Heavy Chain sequences with ESM-2...")
unique_h = seq_df['HAA'].dropna().unique()
h_esm_scores = {seq: compute_esm2_perplexity(seq) for seq in unique_h}

print("Evaluating Light Chain sequences with ESM-2...")
unique_l = seq_df['LAA'].dropna().unique()
l_esm_scores = {seq: compute_esm2_perplexity(seq) for seq in unique_l}

seq_df['Heavy_ESM2_Perplexity'] = seq_df['HAA'].map(h_esm_scores)
seq_df['Light_ESM2_Perplexity'] = seq_df['LAA'].map(l_esm_scores)

print("\nESM-2 Scoring Complete!")
print(seq_df[['HAA', 'Heavy_ESM2_Perplexity', 'LAA', 'Light_ESM2_Perplexity']].dropna().head())

/Users/newlanr1/antibody_engineering/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading Meta ESM-2 transformer model...


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Evaluating Heavy Chain sequences with ESM-2...
Evaluating Light Chain sequences with ESM-2...

ESM-2 Scoring Complete!
                                                 HAA  Heavy_ESM2_Perplexity  \
0  AASGSSGGSSSGAELQLQESGPGLEWIGYIYYSGSTNYNPSLKSLV...                  1.459   
1  CTEAPGKGLEWVANIKQDGSEKYYADSVKGRFTISRDNSKNTLYLQ...                  1.458   
2  DVQLVQSGAEVKKPGASVKVSCKASGYSFTSYYMHWVRQAPGQGLE...                  1.429   
3  DVQLVQSGAEVKKPGASVKVSCKASGYTFISYAISWVRQAPGQGLE...                  1.523   
4  DVQLVQSGAEVKKPGASVKVSCKASGYTFTSYYMHWVRQAPGQGLE...                  1.439   

                                                 LAA  Light_ESM2_Perplexity  
0  QSALTQPASVSGSPGQSITISCTGTSSDVGSYDLVSWYQQHPGKAP...                  1.390  
1  DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSSNKNHLAWYQQKP...                  1.411  
2  QSVLTQPPSVSAAPGQKVTISCSGSSSNIGNYYVSWYQQLPGTAPK...                  1.414  
3  DIVMTQSPLSLPVTPGEPASISCRASQSVYSNYLAWYQQKPGQAPR...                  1.384  
4  DIVMTQSPLSLPV

ML Diagnostics Plotting & PowerPoint Compilation
Description: Plots the relationship between True Framework Errors and ESM-2 Predicted Perplexity, then appends dedicated executive ML slides into Complete_Library_QC_Summary_v2.pptx.

In [6]:
# Plot ESM-2 Perplexity vs Framework Errors
plt.figure(figsize=(8, 4.5))
sns.scatterplot(
    data=seq_df.dropna(subset=['Heavy_ESM2_Perplexity']),
    x='H_True_Errors',
    y='Heavy_ESM2_Perplexity',
    hue='H_Intended_Mutations',
    palette='viridis',
    s=60
)
plt.title('Heavy Chain: True Errors vs. ESM-2 Predicted Perplexity', fontsize=11, fontweight='bold')
plt.xlabel('True Framework Errors', fontsize=10)
plt.ylabel('ESM-2 Perplexity (Lower = Higher Fitness)', fontsize=10)
plt.axvline(2, color='crimson', linestyle='--', label='QC Error Cutoff (≤2)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='CDR Muts')
plt.tight_layout()
plt.savefig('esm2_perplexity_vs_errors.png', dpi=300, bbox_inches='tight')
plt.close()

# Update Presentation Deck
prs = Presentation('Complete_Library_QC_Summary_v2.pptx')

def add_ml_slide(prs, title_text, bullet_points):
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    title_shape = slide.shapes.title
    title_shape.text = title_text
    title_shape.text_frame.paragraphs[0].font.size = Pt(22)
    title_shape.text_frame.paragraphs[0].font.bold = True
    
    tf = slide.shapes.placeholders[1].text_frame
    tf.word_wrap = True
    tf.margin_left, tf.margin_right = Inches(0.5), Inches(0.5)
    tf.margin_top, tf.margin_bottom = Inches(0.2), Inches(0.2)
    
    tf.text = bullet_points[0]
    tf.paragraphs[0].font.size = Pt(14)
    tf.paragraphs[0].space_after = Pt(8)
    
    for pt in bullet_points[1:]:
        p = tf.add_paragraph()
        p.text = pt
        p.font.size = Pt(14)
        p.space_after = Pt(8)

ml_overview_pts = [
    "• KL-Divergence Design Bias: Mathematically quantifies drift between intended trimer probabilities and observed NGS frequencies.",
    "• ESM-2 Transformer Language Model: Evaluates biological compatibility and structural fitness using Meta's ESM-2 model.",
    "• Framework Penalty Detection: Unallowed framework errors correlate directly with increased sequence perplexity (reduced stability).",
    "• Predictive Variant Ranking: Enables prioritization of low-perplexity, high-fitness variants prior to wet-lab screening."
]
add_ml_slide(prs, "Machine Learning: Design Bias & ESM-2 Fitness Scoring", ml_overview_pts)

# Add Visual Slide
slide_img = prs.slides.add_slide(prs.slide_layouts[5])
slide_img.shapes.title.text = "ESM-2 Sequence Fitness vs. Framework Errors"
slide_img.shapes.title.text_frame.paragraphs[0].font.size = Pt(18)
slide_img.shapes.title.text_frame.paragraphs[0].font.bold = True
slide_img.shapes.add_picture('esm2_perplexity_vs_errors.png', Inches(1.0), Inches(1.5), width=Inches(8.0))

prs.save("Complete_Library_QC_Summary_v2.pptx")
print("PowerPoint deck successfully updated with ML metrics: Complete_Library_QC_Summary_v2.pptx")

PowerPoint deck successfully updated with ML metrics: Complete_Library_QC_Summary_v2.pptx
